In [1]:
import pandas as pd
import geopandas as gpd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

In [22]:
## collecting all of the neccesary data
df_Ageo_top20 = pd.read_csv("datasets_rq4/df_Ageo_top20.csv")
df_AT_top20_month = pd.read_csv("datasets_rq4/df_AT_top20_month.csv")
df_with_monthly_delay_rates = pd.read_csv("datasets_rq4\df_with_monthly_delay_rates.csv")
# df_rq1m = pd.read_csv("datasets_rq4/.....")
# df_rq1t = pd.read_csv("datasets_rq4/.....")
# df_rq2m = pd.read_csv("datasets_rq4/.....")
# df_rq2t = pd.read_csv("datasets_rq4/.....")
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
print(df_AT_top20_month.head(1))
df_with_monthly_delay_rates.head(1)


   Unnamed: 0 APT_ICAO  YEAR  MONTH_NUM    FLT_MEAN  FLT_MEAN_DEP  \
0           0     EDDF  2023          1  958.419355    479.967742   

   FLT_MEAN_ARR  
0    478.451613  


<>:4: SyntaxWarning:

invalid escape sequence '\d'

<>:4: SyntaxWarning:

invalid escape sequence '\d'

C:\Users\Bas\AppData\Local\Temp\ipykernel_11388\1328113479.py:4: SyntaxWarning:

invalid escape sequence '\d'



,Unnamed: 0,APT_ICAO,APT_NAME,STATE_NAME,YEAR_MONTH,Mean_delayed_flights,Mean_delay_minutes,YEAR,Total_flights,Total_delayed_flights,Delay_rate_flights
0,0,EDDF,Frankfurt,Germany,2023-01,91.5,2523.5,2023,14832,183.0,1.233819


Merging all datasets to one

In [24]:
df_AT_top20_month['month_year'] = df_AT_top20_month['YEAR'].astype(str) + '-' + df_AT_top20_month['MONTH_NUM'].astype(str).str.zfill(2)

# merging df_Ageo and df_AT
df_total0 = pd.merge(df_AT_top20_month, df_Ageo_top20,
                    left_on='APT_ICAO', right_on='ident', how='left')

# adding df_with_monthly_delay_rates
df_total1 = pd.merge(df_total0, df_with_monthly_delay_rates,
                     left_on=["APT_ICAO", "month_year"], right_on=["APT_ICAO", "YEAR_MONTH"], how="left")

df_total1


,Unnamed: 0_x,APT_ICAO,YEAR_x,MONTH_NUM,FLT_MEAN,FLT_MEAN_DEP,FLT_MEAN_ARR,month_year,Unnamed: 0_y,ident,...,Unnamed: 0,APT_NAME,STATE_NAME,YEAR_MONTH,Mean_delayed_flights,Mean_delay_minutes,YEAR_y,Total_flights,Total_delayed_flights,Delay_rate_flights
0,0,EDDF,2023,1,958.419355,479.967742,478.451613,2023-01,0,EDDF,...,0,Frankfurt,Germany,2023-01,91.500000,2523.500000,2023,14832,183.0,1.233819
1,1,EDDF,2023,2,970.535714,485.357143,485.178571,2023-02,0,EDDF,...,1,Frankfurt,Germany,2023-02,23.000000,417.000000,2023,13585,161.0,1.185131
2,2,EDDF,2023,3,1036.129032,518.096774,518.032258,2023-03,0,EDDF,...,2,Frankfurt,Germany,2023-03,58.333333,929.333333,2023,16059,525.0,3.269195
3,3,EDDF,2023,4,1183.433333,591.333333,592.100000,2023-04,0,EDDF,...,3,Frankfurt,Germany,2023-04,48.142857,720.428571,2023,17763,337.0,1.897202
4,4,EDDF,2023,5,1202.193548,600.935484,601.258065,2023-05,0,EDDF,...,4,Frankfurt,Germany,2023-05,218.500000,5208.692308,2023,18639,5681.0,30.479103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,475,LTFM,2024,8,1500.258065,750.290323,749.967742,2024-08,19,LTFM,...,475,Istanbul,Türkiye,2024-08,99.333333,4485.333333,2024,23249,298.0,1.281776
476,476,LTFM,2024,9,1480.800000,740.166667,740.633333,2024-09,19,LTFM,...,476,Istanbul,Türkiye,2024-09,57.000000,1965.200000,2024,22219,285.0,1.282686
477,477,LTFM,2024,10,1364.354839,682.290323,682.064516,2024-10,19,LTFM,...,477,Istanbul,Türkiye,2024-10,83.000000,1892.000000,2024,21144,83.0,0.392546
478,478,LTFM,2024,11,1339.166667,669.433333,669.733333,2024-11,19,LTFM,...,478,Istanbul,Türkiye,2024-11,114.500000,4523.500000,2024,20092,229.0,1.139757


## Code for the map

In [28]:
min_delay = df_total1['Mean_delayed_flights'].min()
max_delay = df_total1['Mean_delayed_flights'].max()


map_fig = px.scatter_map(
    df_total1,
    lat="latitude_deg",
    lon="longitude_deg",
    size="FLT_MEAN",
    color="Mean_delayed_flights",
    color_continuous_scale=["green", "yellow", "red"],
    range_color=[min_delay, max_delay],
    hover_name="name",
    map_style="carto-positron",
    zoom=4,
    width=1000,
    height=700,
    animation_frame="month_year",
    size_max=35
)

map_fig.show()


